# 04 — Identifiability

**Phenomenological Growth Models Lab**

In notebook 03 the interval on $a$ came out enormous while the fitted curve looked
excellent. That is not a bug, and more data of the same kind will not fix it.

The issue is **practical identifiability**: whether the data can distinguish the
parameters from one another at all. Two parameters can trade off so precisely that many
different combinations produce nearly the same curve. The fit is fine; the individual
numbers are not interpretable.

Assumes [03 — Uncertainty](03_uncertainty.ipynb).

## Setup

These notebooks run against the environment pinned in this repository's `Project.toml`
and `Manifest.toml`. The cell below activates it and installs anything missing.

The first run downloads and precompiles the dependencies — a few minutes, once.
Every run after that starts in seconds.

Because the versions are pinned in `Manifest.toml`, you get the same package versions
these notebooks were written against, whatever else is installed on your machine. If you
skip this cell and rely on your global environment, newer versions may resolve
differently and the numbers below may not match.

In [ ]:
using Pkg

# Works whether the notebook was launched from the repository root or from notebooks/.
root = isfile("Project.toml") ? "." : ".."
Pkg.activate(root)
Pkg.instantiate()

println("Active project: ", Base.active_project())

In [ ]:
using GrowthFit
using Plots, Statistics, Printf, DelimitedFiles, Random

default(size = (760, 420), lw = 2, legend = :topright, framestyle = :box,
        guidefontsize = 10, titlefontsize = 11)

# Threads. Notebooks 03 and 05 bootstrap in parallel. A Jupyter kernel does
# NOT inherit `-t` from whatever launched Jupyter — the thread count is fixed
# by the kernel itself — so if this says 1, see the README for the one-off
# `IJulia.installkernel` step. Results are identical either way; only speed
# changes.
println("GrowthFit loaded. Threads available: ", Threads.nthreads())
Threads.nthreads() == 1 && println("  (single-threaded — notebooks 03 and 05 will be slower; see README)")


> **Tip.** The bootstrap in notebooks 03 and 05 runs in parallel. If you want it faster,
> restart Julia with `-t 8`. Results are identical either way — each replicate is seeded
> independently of the thread count.

In [ ]:
function load_series(name)
    path = isfile(joinpath("data", name)) ? joinpath("data", name) :
           joinpath("..", "data", name)
    isfile(path) || error("Could not find $name. Run this notebook from inside the " *
                          "cloned repository so the data/ folder is reachable.")
    raw = readdlm(path, '\t', Float64)
    return raw[:, 1], raw[:, 2]          # time, incidence
end

timevect_all, ydata_all = load_series("jalisco_measles.txt")

@printf("%d weeks, peak of %.0f cases, %.0f cases total\n",
        length(ydata_all), maximum(ydata_all), sum(ydata_all))

In [ ]:
calibration_period = 40
forecast_horizon   = 3
timevect = timevect_all[1:calibration_period]
ydata    = ydata_all[1:calibration_period]
I0 = ydata[1]

## Seeing the trade-off directly

The bootstrap already contains the evidence. If two parameters compensate for each
other, their bootstrap replicates will be correlated — high $p$ paired with low $a$,
and so on.

In [ ]:
Random.seed!(1)
fit  = fit_growth_model(:grm, timevect, ydata; dist = :nb1, n_restarts = 10)
boot = run_bootstrap(:grm, timevect, ydata, fit;
                     dist = :nb1, M = 300, forecast_horizon = 3, seed = 20260726)

pnames = ["r", "p", "a", "K"]
P = boot.params[:, 1:4]

println("Correlation between bootstrap parameter estimates")
println()
print("        ")
for nm in pnames; @printf("%8s", nm); end
println()
for i in 1:4
    @printf("%-8s", pnames[i])
    for j in 1:4
        @printf("%8.2f", cor(P[:, i], P[:, j]))
    end
    println()
end

Anything above about 0.9 in absolute value is a red flag: those two parameters are
nearly interchangeable as far as this dataset is concerned.

The scatter plots make it concrete:

In [ ]:
pairs = [(1,2), (1,3), (2,3), (3,4)]
plts = [scatter(P[:, i], P[:, j], ms = 2, msw = 0, alpha = 0.4, label = "",
                xlabel = pnames[i], ylabel = pnames[j],
                title = @sprintf("cor = %.2f", cor(P[:, i], P[:, j])),
                titlefontsize = 9, mc = :steelblue)
        for (i, j) in pairs]
plot(plts..., layout = (2, 2), size = (820, 520))

A tight diagonal cloud means a ridge in the likelihood surface: you can move along it
in either direction and the fit barely changes. A round blob means the two parameters
are being estimated independently, which is what you want.

## Different parameters, same curve

Take two bootstrap replicates from opposite ends of the ridge and plot their curves.
The parameters differ substantially; the trajectories barely do.

In [ ]:
# Order replicates along the p-a ridge and take the extremes
ord = sortperm(P[:, 2])
lo_i, hi_i = ord[10], ord[end - 9]

for (lab, i) in (("low p ", lo_i), ("high p", hi_i))
    @printf("%s: r=%.3f p=%.3f a=%.3f K=%.0f\n",
            lab, P[i,1], P[i,2], P[i,3], P[i,4])
end

c_lo = solve_incidence(:grm, P[lo_i,1], P[lo_i,2], P[lo_i,3], P[lo_i,4], I0, timevect)
c_hi = solve_incidence(:grm, P[hi_i,1], P[hi_i,2], P[hi_i,3], P[hi_i,4], I0, timevect)

plot(timevect, c_lo, label = "low p, compensating a", lc = :crimson)
plot!(timevect, c_hi, label = "high p, compensating a", lc = :seagreen, ls = :dash)
scatter!(timevect, ydata, label = "observed", mc = :grey55, ms = 3, msw = 0,
         xlabel = "week", ylabel = "weekly cases",
         title = "Different parameters, nearly identical curve")

If the two lines sit almost on top of each other, the data cannot tell those parameter
sets apart. Reporting $p = 0.84$ with a confident interpretation would be overclaiming.

## The collinearity index

There is a formal version of this that does not require a bootstrap.
`screen_identifiability` perturbs each parameter slightly, records how the trajectory
responds, and measures how close those sensitivity patterns are to being linearly
dependent. If moving $p$ up can be almost exactly undone by moving $a$ down, the
sensitivity vectors are nearly collinear and the index is large.

The conventional threshold is **20** (Omlin & Reichert, 2001). Below that, the
parameters are considered practically identifiable; above, treat them with caution.

In [ ]:
θ = (r = fit.r, p = fit.p_exp, a = fit.a, K = fit.K, I0 = I0)
out = screen_identifiability(:grm, timevect, θ)

println("free parameters   : ", out.free_names)
@printf("collinearity index: %.2f\n", out.collinearity)
println("identifiable      : ", out.identifiable)
println("worst pair        : ", out.worst_pair)
println("note              : ", out.note)

### Across all models

More parameters means more room for them to trade off. The pattern is usually monotone:

In [ ]:
screen_all_models(timevect, ydata)

Simple models are identifiable and inflexible. Flexible models fit better and mean less.
That tension does not go away — it is the reason model selection in notebook 02 is not
just a matter of picking the lowest AICc.

## Does more data help?

Sometimes. It depends on *which* data. A longer series through the peak and into the
decline is far more informative about $K$ and $a$ than the same number of extra points
during exponential growth, because the turnover is what those parameters describe.

In [ ]:
println("weeks   collinearity   K estimate")
println("-"^40)
for n in (20, 25, 30, 35, 40)
    tv, yd = timevect_all[1:n], ydata_all[1:n]
    Random.seed!(1)
    f = fit_growth_model(:grm, tv, yd; dist = :nb1, n_restarts = 10)
    θn = (r = f.r, p = f.p_exp, a = f.a, K = f.K, I0 = yd[1])
    o = screen_identifiability(:grm, tv, θn)
    @printf("%5d %14.2f %12.0f\n", n, o.collinearity, f.K)
end

Watch both columns. The collinearity index should fall as the window extends past the
peak, and the estimate of $K$ should stabilize. If $K$ is still moving substantially at
40 weeks, it was never well determined.

## What to do about it

Practical identifiability is a property of *model plus data*, not of the model alone.
Your options, roughly in order of preference:

1. **Use a simpler model.** If $p$ and $a$ cannot be separated, fit the Generalized
   Logistic instead of the Generalized Richards. You lose flexibility you were not
   actually able to use.

2. **Report the derived quantity, not the parameters.** $K$ and the peak timing are
   often well determined even when the individual shape parameters are not. Report what
   the data supports.

3. **Fix a parameter at a defensible value** from other studies, and say you did.

4. **Wait for more data** — the honest answer in real time, and the least satisfying.

What *not* to do is report a tight-looking point estimate for a parameter sitting on a
ridge. The optimizer will happily give you one.

### A caveat about the diagnostic itself

`screen_identifiability` is **local**. It evaluates sensitivities around one point in
parameter space, so it describes the surface near your fit — not globally. A model can
look identifiable at the optimum and be badly behaved elsewhere. The bootstrap
correlations you started with are a useful complement precisely because they explore a
wider region.

---

## Exercises

1. Run the correlation analysis on the Generalized Logistic instead. Are $p$ and $K$
   correlated once $a$ is removed from the model?

2. Repeat the whole notebook on the 25-week (pre-peak) window. Which pairs become
   worse, and does that match your intuition about what the turnover tells you?

3. Simulate data from a known GRM with `solve_incidence`, add negative-binomial noise,
   and refit. Does the bootstrap interval actually contain the true $a$ you used?

**Next:** [05 — Forecasting](05_forecasting.ipynb).